In [ ]:
import pandas as pd
import glob
import os
import numpy as np
import matplotlib.pyplot as plt

# =====================================================
# ROOT CONFIG (CONVOLUTIONAL FMNIST)
# =====================================================
ROOT_DIR = r"C:\Users\Student\Desktop\Neural_research\physlab\Convolution\Convolutional\Convolutional-FMNIST"

PRUNE_TYPES = [
    "prune_layers_ALL",
    "prune_layers_CONV",
    "prune_layers_FHL",
    "prune_layers_SHL",
    "prune_layers_FHL+SHL",
]

BATCH_DIR_TEMPLATE = "p-percentage_{:.1f}\\batch_size_{}"
FILE_PATTERN = "convol_{:.1f}_{}_run_*.txt"

BATCH_SIZES = [64, 1024]
PRUNING_LEVELS = [round(x * 0.1, 1) for x in range(11)]

PLOT_ROOT = os.path.join(ROOT_DIR, "Avg_Plots")

# =====================================================
# COLOR LIST (matches SLP/FMNIST)
# =====================================================
COLOR_LIST = [
    "#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#800080",
    "#8c564b", "#e377c2", "#7f7f7f", "#bcbd22", "#B9D9EB", "#17becf"
][::-1]  # reversed so P%=0 -> teal, P%=100 -> blue

# =====================================================
# STYLE (Nature-like)
# =====================================================
plt.rcParams.update({
    "font.size": 18,
    "axes.titlesize": 18,
    "axes.labelsize": 18,
    "xtick.labelsize": 18,
    "ytick.labelsize": 18,
    "legend.fontsize": 15
})

LN10 = np.log(10)

# =====================================================
# MAIN LOOP
# =====================================================
for prune_type in PRUNE_TYPES:

    print(f"\n=== Processing {prune_type} ===")

    BASE_DIR = os.path.join(ROOT_DIR, prune_type)
    PLOT_DIR = os.path.join(PLOT_ROOT, prune_type)
    os.makedirs(PLOT_DIR, exist_ok=True)

    for bs in BATCH_SIZES:
        all_avg_dfs = {}

        # ==============================================
        # LOAD + AVERAGE RUNS (ALL PRUNING LEVELS)
        # ==============================================
        for p in PRUNING_LEVELS:
            folder = os.path.join(BASE_DIR, BATCH_DIR_TEMPLATE.format(p, bs))
            files = glob.glob(os.path.join(folder, FILE_PATTERN.format(p, bs)))

            if not files:
                print(f"[WARNING] No files for {prune_type}, p={p}, bs={bs}")
                continue

            dfs = []
            for f in files:
                df = pd.read_csv(f, sep=r"\s+")
                df.columns = df.columns.str.strip()
                df["CE_Train"] = pd.to_numeric(df["CE_Train"], errors="coerce")
                df["CE_TEST"] = pd.to_numeric(df["CE_TEST"], errors="coerce")
                df["Accuracy(%)"] = pd.to_numeric(df["Accuracy(%)"], errors="coerce")
                dfs.append(df)

            all_runs = pd.concat(dfs, ignore_index=True)

            avg_df = all_runs.groupby("Batch_Number", as_index=False).agg(
                Avg_CE_Train=("CE_Train", "mean"),
                Avg_CE_Test=("CE_TEST", "mean"),
                Avg_Accuracy=("Accuracy(%)", "mean"),
                Num_Runs=("CE_TEST", "count")
            )

            # filename matches MNIST convention: averaged_runs_conv_{prune_layer}_p_{p}_bs_{bs}.csv
            short_name = prune_type.split("prune_layers_")[-1]
            avg_csv = os.path.join(folder, f"averaged_runs_conv_{short_name}_p_{p}_bs_{bs}.csv")
            avg_df.to_csv(avg_csv, index=False)

            all_avg_dfs[p] = avg_df

        if not all_avg_dfs:
            continue

        pretty_prune = prune_type.split("prune_layers_")[-1]
        if pretty_prune == "ALL":
            pretty_prune = pretty_prune + "-Layers"

        # ==============================================
        # COMBINED PLOTS
        # ==============================================
        def plot_metric(metric_col, ylabel, title_text, subtitle_text, filename, legend_loc, bbox):
            plt.figure(figsize=(12, 6))

            for idx, (p, df) in enumerate(sorted(all_avg_dfs.items())):
                color = COLOR_LIST[idx % len(COLOR_LIST)]
                plt.plot(df["Batch_Number"], df[metric_col],
                         label=f"P%={int(p * 100)}", color=color)

            plt.xlabel("Batch Number")
            plt.ylabel(ylabel)

            if metric_col in ["Avg_CE_Train", "Avg_CE_Test"]:
                plt.ylim(0, 2.5)
                plt.text(0.06, LN10 + 0.05, r"$\ln(10)$",
                         transform=plt.gca().get_yaxis_transform(),
                         fontsize=14, va="center", ha="left")
                plt.gca().yaxis.set_minor_locator(plt.FixedLocator([LN10]))
                plt.tick_params(axis='y', which='minor', length=5, color='black')
                plt.text(0.5, 0.95, title_text, ha="center",
                         transform=plt.gca().transAxes, va="top")
                plt.text(0.5, 0.90, subtitle_text, ha="center",
                         fontsize=16, transform=plt.gca().transAxes, va="top")
            else:
                plt.ylim(0, 100)
                plt.text(0.5, 0.55, title_text, ha="center",
                         transform=plt.gca().transAxes, va="top")
                plt.text(0.5, 0.50, subtitle_text, ha="center",
                         fontsize=16, transform=plt.gca().transAxes, va="top")

            plt.legend(loc=legend_loc, bbox_to_anchor=bbox, frameon=False)
            plt.grid(True)
            plt.tight_layout()
            plt.savefig(os.path.join(PLOT_DIR, filename), dpi=300, bbox_inches="tight")
            plt.show()

        # ---- CE TRAIN ----
        plot_metric(
            metric_col="Avg_CE_Train",
            ylabel="Average CE",
            title_text="Cross-Entropy",
            subtitle_text=f"(CNN-{pretty_prune} FMNIST Training-Vectors, BS={bs})",
            filename=f"CE_Train_Avg_CONV_FMNIST_{pretty_prune}_BS_{bs}.png",
            legend_loc="upper right",
            bbox=(1.0, 1.02)
        )

        # ---- CE TEST ----
        plot_metric(
            metric_col="Avg_CE_Test",
            ylabel="Average CE",
            title_text="Cross-Entropy",
            subtitle_text=f"(CNN-{pretty_prune} FMNIST Test-Vectors, BS={bs})",
            filename=f"CE_Test_Avg_CONV_FMNIST_{pretty_prune}_BS_{bs}.png",
            legend_loc="upper right",
            bbox=(1.0, 1.02)
        )

        # ---- ACCURACY ----
        plot_metric(
            metric_col="Avg_Accuracy",
            ylabel="Average Accuracy",
            title_text="Accuracy",
            subtitle_text=f"(CNN-{pretty_prune} FMNIST Test-Vectors, BS={bs})",
            filename=f"Accuracy_Avg_CONV_FMNIST_{pretty_prune}_BS_{bs}.png",
            legend_loc="lower right",
            bbox=(1.0, 0.1)
        )

        print(f"[DONE] {prune_type}, BS={bs}")